## Simple scenario: predict validation `per_sample` parameters through a model (Linear/RandomForest) trained on input features (`cytof_init`) -> parameter deviations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor

from common import MODEL_FEATURE_PREFIX, training_samples, val_samples, PER_SAMPLE_OUTFILE_PARS
from dmm import MEDIAN_FEATURE_PREFIX
from dmm.config_options import Conf
from dmm.feature_selection import load_data
from util import load_petab_base_files
from dmm.petab_subproblem import load_petab
from cytof.problem import CytofProblem
from evaluation_utils import generate_per_sample_pretraining_problems
from amici.petab.simulations import rdatas_to_simulation_df
from dmm.analysis import process_simulation

# Set random seed
np.random.seed(42)

# === Stage 1: Load per-sample pretraining parameter sets ===
conf = Conf(model="EGFR_MAPK", data="dream_cytof", samples="0of5")

samples_train, samples_val = training_samples(conf), val_samples(conf)
sample_name_list = samples_train + samples_val

per_sample_parameter_file = PER_SAMPLE_OUTFILE_PARS.format(
    **{**conf.__dict__, "sample": "{sample}"}
)

pretrained_samples = {}

for sample in sample_name_list:
    df = pd.read_csv(
        per_sample_parameter_file.format(sample=sample),
        index_col=[0],
    )
    pretrained_samples[sample] = df[
        [
            col
            for col in df.columns
            if not col.startswith(MODEL_FEATURE_PREFIX)
        ]
    ]

# Poisson sample multistart
par_combo = pd.concat(
    [
        pretraining[
            pretraining.index
            == np.min(
                [np.random.poisson(2, 1)[0], len(pretraining) - 1]
            )
        ]
        for pretraining in pretrained_samples.values()
    ]
)
par_combo.rename(
    columns=lambda col: col[len(MEDIAN_FEATURE_PREFIX) :]
    if col.startswith(MEDIAN_FEATURE_PREFIX)
    else col,
    inplace=True,
)
par_combo.index = list(pretrained_samples.keys())
par_combo = par_combo.reindex(sample_name_list)

# === Define observable scaling parameters ===
obs_scaling_cols = [
    "pERBB2_Y1248_obs_offset", "pERBB2_Y1248_obs_scale",
    "pERK_Y204_obs_offset", "pERK_Y204_obs_scale",
    "pMEK_S222_obs_offset", "pMEK_S222_obs_scale",
]

for col in obs_scaling_cols:
    par_combo[col].fillna(1.0, inplace=True)

# === Compute global nanmean of scaling parameters from par_combo ===
scaling_values = par_combo[obs_scaling_cols].copy()
scaling_means = scaling_values.apply(np.nanmean).to_dict()

# Print computed means
print("Observable scaling means:")
for k, v in scaling_means.items():
    print(f"  {k}: {v:.4f}")

# === Drop all-NaN columns (optional cleanup before column filtering) ===
par_combo.dropna(axis=1, inplace=True)

# === Remove scaling parameters from par_combo ===
# par_combo.drop(columns=[c for c in par_combo.columns if c in obs_scaling_cols], inplace=True)

# Final check
print(f"Final par_combo shape (regressor inputs only): {par_combo.shape}")

# Load PEtab base once
petab_base_files_all = load_petab_base_files(conf)
del petab_base_files_all["condition_table"]

rmse_dict = {}

# === Stage 2: Train RF per split, simulate predictions ===
for split in [f"{i}of5" for i in range(5)]:
    print(f"\n===== CV split: {split} =====")
    conf = Conf(model="EGFR_MAPK", data="dream_cytof", samples=split)
    samples_train, samples_val = training_samples(conf), val_samples(conf)
    rmse_dict[split] = {}

    train_targets = par_combo[par_combo.index.isin(samples_train)]
    val_targets = par_combo[par_combo.index.isin(samples_val)]

    input_train, features_train = load_data(
        contextualization="cytof_init",
        samples=samples_train,
        features=None,
        **petab_base_files_all,
    )

    input_val, _ = load_data(
        contextualization="cytof_init",
        samples=samples_val,
        features=features_train,
        **petab_base_files_all,
    )

    rf = RandomForestRegressor()
    rf.fit(input_train.values, train_targets.values)

    train_pred = rf.predict(input_train.values)
    val_pred = rf.predict(input_val.values)

    train_rmse = np.sqrt(np.mean((train_pred - train_targets.values) ** 2))
    val_rmse = np.sqrt(np.mean((val_pred - val_targets.values) ** 2))
    print(f"Training RMSE: {train_rmse:.3f}, Validation RMSE: {val_rmse:.3f}")

    rmse_dict[split]["params_train"] = train_rmse
    rmse_dict[split]["params_val"] = val_rmse

    # === Stage 3: Simulate predictions ===
    ref_conf = Conf(model=conf.model, data=conf.data, max_lrate=0, lrate_span=0, lrate_decay=0)
    petab_base_files = load_petab_base_files(conf)
    problem = CytofProblem(conf.model)
    petab_base_importer = load_petab(problem, conf.data, **petab_base_files)

    for sample_set, prediction, set_name in [
        (samples_train, train_pred, "train"),
        (samples_val, val_pred, "val"),
    ]:
        evaluations = []

        for i, sample in enumerate(sample_set):
            importer = generate_per_sample_pretraining_problems(
                petab_base_importer,
                problem,
                conf.data,
                sample,
            )
            problem_sample = importer.create_problem()
            problem.apply_objective_settings(problem_sample.objective)

            pred_df = pd.DataFrame(
                prediction[i:i+1],
                columns=par_combo.columns,
            )

            # # Inject observable scalings
            # for col, value in scaling_means.items():
            #     pred_df[col] = value

            x = problem_sample.get_reduced_vector(
                pred_df.values.reshape(-1, ), problem_sample.x_free_indices
            )

            res = problem_sample.objective(x, return_dict=True)

            simulation_df = rdatas_to_simulation_df(
                res["rdatas"],
                model=problem_sample.objective.amici_model,
                measurement_df=importer.petab_problem.measurement_df,
            )

            process_simulation(
                evaluations=evaluations,
                measurement_df=importer.petab_problem.measurement_df,
                simulation_df=simulation_df,
                conf=ref_conf,
                sample=sample,
            )

        eval_df = pd.DataFrame(evaluations)
        overall_rmse = np.sqrt(np.mean(np.square(eval_df["res"])))
        print(f"{set_name.capitalize()} simulation RMSE: {overall_rmse:.3f}")
        rmse_dict[split][set_name] = overall_rmse

        if set_name == "val":  # Only plot for validation samples
            for i, sample in enumerate(sample_set):
                importer = generate_per_sample_pretraining_problems(
                    petab_base_importer,
                    problem,
                    conf.data,
                    sample,
                )
                problem_sample = importer.create_problem()
                problem.apply_objective_settings(problem_sample.objective)

                pred_df = pd.DataFrame(
                    prediction[i:i+1],
                    columns=par_combo.columns,
                )

                # # Inject observable scalings
                # for col, value in scaling_means.items():
                #     pred_df[col] = value

                x = problem_sample.get_reduced_vector(
                    pred_df.values.reshape(-1, ), problem_sample.x_free_indices
                )

                res = problem_sample.objective(x, return_dict=True)

                # === Convert to PEtab simulation DataFrame ===
                sim_df = rdatas_to_simulation_df(
                    res["rdatas"],
                    model=problem_sample.objective.amici_model,
                    measurement_df=importer.petab_problem.measurement_df,
                )
                sim_df["source"] = "simulation"
                sim_df.rename(columns={"simulation": "value"}, inplace=True)

                # === Prepare measurement DataFrame ===
                meas_df = importer.petab_problem.measurement_df.copy()
                meas_df["value"] = meas_df["measurement"]
                meas_df["source"] = "measurement"

                # === Merge for overlay plotting ===
                combined = pd.concat([
                    meas_df[["time", "observableId", "simulationConditionId", "value", "source"]],
                    sim_df[["time", "observableId", "simulationConditionId", "value", "source"]],
                ])

                g = sns.FacetGrid(
                    combined,
                    col="observableId",
                    row="simulationConditionId",
                    margin_titles=True,
                    height=2.5,
                    sharey=False,
                )
                g.map_dataframe(
                    sns.lineplot,
                    x="time",
                    y="value",
                    hue="source",
                    palette={"measurement": "blue", "simulation": "red"},
                )
                g.add_legend()
                g.fig.suptitle(f"Validation Overlay for Sample: {sample}", y=1.02)
                plt.show()

Observable scaling means:
  pERBB2_Y1248_obs_offset: 0.4975
  pERBB2_Y1248_obs_scale: 1.7081
  pERK_Y204_obs_offset: 0.3665
  pERK_Y204_obs_scale: 2.2122
  pMEK_S222_obs_offset: 0.5009
  pMEK_S222_obs_scale: 1.6467
Final par_combo shape (regressor inputs only): (52, 24)

===== CV split: 0of5 =====
Training RMSE: 0.685, Validation RMSE: 1.408


In [33]:
rmse_dict

{'0of5': {'train': 0.7600156535166486, 'val': 1.1559890099258407},
 '1of5': {'train': 0.9737116857488424, 'val': 1.1797424664770573},
 '2of5': {'train': 1.0017098393833088, 'val': 0.9411776847482958},
 '3of5': {'train': 0.9729013960122669, 'val': 0.7512944065721088},
 '4of5': {'train': 0.9258361715813661, 'val': 0.7586953647846709}}